In [4]:
!pip install anthropic

import anthropic

import json

In [10]:
key = '<KEY>'

# Set the Key
client = anthropic.Anthropic(api_key=key)

tools = [
    {
        'name' : 'add',
        'description' : 'A function to add numbers',
        'input_schema' : {
            'type' : 'object',
            'properties' : {
                'a' : {'type' : 'number'},
                'b' : {'type' : 'number'}
            },
            'required' : ['a', 'b']
        }
    },
    {
        'name' : 'sub',
        'description' : 'the function to subtract two numbers',
        'input_schema' : {
            'type' : 'object',
            'properties' : {
                'a' : {'type' : 'number'},
                'b' : {'type' : 'number'}
            },
            'required' : ['a', 'b']
        }
    },
    {
        'name' : 'mul',
        'description' : 'multiply two numbers',
        'input_schema' : {
            'type' : 'object',
            'properties' : {
                'a' : {'type' : 'number'},
                'b' : {'type' : 'number'}
            },
            'required' : ['a', 'b']
        }
    }
 ]

def add(a, b):
  return a + b

def sub(a, b):
  return a - b

def mul(a, b):
  return a * b

def run_tool(name, input):

  if name == 'add':
    return add(input['a'], input['b'])
  elif name == 'sub':
    return sub(input['a'], input['b'])
  elif name == 'mul':
    return mul(input['a'], input['b'])
  else:
    return ValueError("The tool does not exist.")

messages = [
    {
      'role': 'user',
      'content' : 'What is 2 + 2 * 4 - 7?'
    }
]

response = client.messages.create(
    model = 'claude-opus-4-6',
    max_tokens = 1024,
    tools = tools,
    messages = messages
)

# If the response requiresa tool use, then loop until it doesn't
while response.stop_reason == 'tool_use':

  tool_results = []

  for block in response.content:
    if block.type == 'tool_use':

      result = run_tool(block.name, block.input)
      tool_results.append({
          'type' : 'tool_result',
          'tool_use_id' : block.id,
          'content' : json.dumps(result)
      })

      # After running the tool, now append to the message array
      # and query communicte with Claude again

      messages.append({
          'role' : 'assistant',
          'content' : response.content
      })
      messages.append({
          'role' : 'user',
          'content' : tool_results
      })

      # Now send message to claude again
      response = client.messages.create(
          model = 'claude-opus-4-6',
          max_tokens = 1024,
          tools = tools,
          messages = messages
      )

# Once Claude responds without a tool_use response, it means we hve stepped out of the Agent Loop
final_text = next(block for block in response.content if block.type == 'text')
print(final_text.text)






**2 + 2 × 4 - 7 = 3**

Here's the breakdown using order of operations (PEMDAS):
1. **Multiply first:** 2 × 4 = 8
2. **Then add:** 2 + 8 = 10
3. **Then subtract:** 10 - 7 = **3**
